# Notebook 11 — String Functions Basics

**Datasets:** `samples.bakehouse.sales_customers`, `samples.tpch.customer`  
**Difficulty:** Easy  
**Topics:** `upper`, `lower`, `trim`, `length`, `concat`, `split`, `substring`, `contains`, `startswith`, `endswith`

Each problem teaches a core PySpark string function through a practical example.

In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = SparkSession.builder.getOrCreate()

customers = spark.read.table("samples.bakehouse.sales_customers")
transactions = spark.read.table("samples.bakehouse.sales_transactions")
tpch_customer = spark.read.table("samples.tpch.customer")

print("sales_customers schema:")
customers.printSchema()
print("sales_transactions schema:")
transactions.printSchema()
print("tpch.customer schema:")
tpch_customer.printSchema()

## Learn — String Functions

| Function | What it does |
|----------|-------------|
| `F.upper(col)` / `F.lower(col)` | Convert to upper/lower case |
| `F.length(col)` | Number of characters in a string |
| `F.trim(col)` / `F.ltrim()` / `F.rtrim()` | Remove leading/trailing whitespace |
| `F.concat(col1, F.lit(" "), col2)` | Concatenate strings (use `F.lit()` for literal text) |
| `F.split(col, pattern)` | Split string into an array by delimiter |
| `F.substring(col, pos, len)` | Extract a substring (1-based index) |
| `F.col("c").contains("text")` | True if string contains the substring |
| `F.col("c").startswith("X")` | True if string starts with the value |

**Docs:** [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) · [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)

> `F.lit("text")` wraps a Python literal as a Spark Column so it can be used inside `concat()` and similar functions.

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.bakehouse.sales_customers")

# upper/lower/concat — using address fields (not name fields, which the problems use)
df.select(
    F.upper(F.col("city")).alias("city_upper"),
    F.lower(F.col("country")).alias("country_lower"),
    F.concat(F.col("city"), F.lit(", "), F.col("country")).alias("location")
).show(5)

# split and index — extract domain from email
df.select(
    F.col("email_address"),
    F.split(F.col("email_address"), "@")[1].alias("domain")
).show(5)

## Problem 1 — Standardise Customer Names

Inconsistent casing is a common data quality issue. On `sales_customers`, use `F.upper()` on `first_name` and `F.lower()` on `last_name`. Concatenate them into a `full_name` column (UPPER_FIRST + space + lower_last) using `F.concat()`.

**Why it matters:** standardising case before joins or comparisons prevents duplicate records.

**Expected output columns:** `customerID`, `full_name`, `upper_first`, `lower_last`

In [0]:
result_1 = customers.select(
    "customerID",
    F.concat(
        F.upper("first_name"),
        F.lit(" "),
        F.lower("last_name")
    ).alias("full_name"),
    F.upper("first_name").alias("upper_first"),
    F.lower("last_name").alias("lower_last")
)

In [0]:
cust_pd = customers.toPandas()
cols_to_string = {col: 'string' for col in cust_pd.columns if col not in ('customerID', 'postal_zip_code')}
cust_pd = cust_pd.astype(cols_to_string)

In [0]:
result_1_pd = (
    cust_pd
    .assign(
        full_name = lambda x: x['first_name'].str.upper() + " " + x['last_name'].str.lower(),
        upper_first = cust_pd['first_name'].str.upper(),
        lower_last = cust_pd['last_name'].str.lower()
    )
    .loc[:, ['customerID', 'full_name', 'upper_first', 'lower_last']]
)

result_1_pd.display()

In [0]:
result_1.show(5)

In [0]:
# ── Tests for Problem 1 ─────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'full_name' in cols, "Missing column: full_name"
assert 'upper_first' in cols, "Missing column: upper_first"
assert 'lower_last' in cols, "Missing column: lower_last"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2 — Email Username Extraction

Split the `email_address` column on `@` to extract the username (part before `@`) and domain (part after `@`). Use `F.split(col, "@")[0]` and `F.split(col, "@")[1]`.


**Expected output columns:** `customerID`, `email_address`, `username`, `domain`

In [0]:
result_2 = customers.select(
    "customerID",
    "email_address",
    F.split("email_address", "@")[0].alias("username"),
    F.split("email_address", "@")[1].alias("domain")
)

In [0]:
result_2_pd = (
    cust_pd
    .loc[:, ['customerID', 'email_address']]
    .assign(
        username = cust_pd['email_address'].str.split('@').str.get(0),
        domain = cust_pd['email_address'].str.split('@').str.get(1)
    )
)

result_2_pd.display()

In [0]:
result_2.show(5)

In [0]:
# ── Tests for Problem 2 ─────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'email_address' in cols, "Missing column: email_address"
assert 'username' in cols, "Missing column: username"
assert 'domain' in cols, "Missing column: domain"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3 — Name Length Analysis

Use `F.length()` to compute `first_name_length` and `last_name_length`. Add a `total_length` column as their sum. Filter to customers whose `total_length` exceeds 20 characters.

**Expected output columns:** `customerID`, `first_name`, `last_name`, `first_name_length`, `last_name_length`, `total_length`

In [0]:
result_3 = customers.select(
    "customerID",
    "first_name",
    "last_name",
    F.length("first_name").alias("first_name_length"),
    F.length("last_name").alias("last_name_length"),
    (F.length("first_name") + F.length("last_name")).alias("total_length")
).orderBy(F.col("total_length").desc())

In [0]:
res_3_pd = (
    cust_pd
    .loc[:, ['customerID', 'first_name', 'last_name']]
    .assign(
        first_name_length = cust_pd['first_name'].str.len(),
        last_name_length = cust_pd['last_name'].str.len(),
        total_length = lambda x: x['first_name'].str.len() + x['last_name'].str.len()
    )
    .sort_values('total_length', ascending=False)
)

res_3_pd.display()

In [0]:
result_3.show(5)

In [0]:
# ── Tests for Problem 3 ─────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'first_name' in cols, "Missing column: first_name"
assert 'last_name' in cols, "Missing column: last_name"
assert 'first_name_length' in cols, "Missing column: first_name_length"
assert 'last_name_length' in cols, "Missing column: last_name_length"
assert 'total_length' in cols, "Missing column: total_length"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4 — Product Name Search

On `sales_transactions`, find all coffee-related products using `.contains("Coffee")` or `.like("%Coffee%")`. Group by `product` and count transactions.


**Expected output columns:** `product`, `transaction_count`

In [0]:
result_4 = transactions.filter(F.col("product").contains("Coffee")).groupBy("product").agg(
    F.count("*").alias("transaction_count")
)

In [0]:
transaction_pd = transactions.toPandas()

In [0]:
transaction_pd.display()

In [0]:

res_4 = (
    transaction_pd
    .loc[transaction_pd['product'] == 'Pearly Pies']
    .groupby('product')
    .agg(
        transaction_count = ('transactionID', 'size')
    )
    .sort_values('transaction_count', ascending=False)
)

res_4.display()

In [0]:
# ── Tests for Problem 4 ─────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'product' in cols, "Missing column: product"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5 — Substring Extraction

On `tpch.customer`, use `F.substring("c_phone", 1, 3)` to extract the first 3 characters of the phone number as a `country_code`. Count customers per country code.


**Expected output columns:** `country_code`, `customer_count`

In [0]:
display(tpch_customer.head(5))

In [0]:
result_5 = tpch_customer.groupBy(
    F.substring("c_phone", 1, 2).alias("country_code")
).agg(F.count("*").alias("customer_count"))

In [0]:
custt_pd = tpch_customer.toPandas()
res_5 = (
    custt_pd
    .assign(
        country_code = custt_pd['c_phone'].str[0:2]
    )
    .groupby('country_code', as_index=False)
    .agg(
        customer_count = ('country_code', 'size')
    )
    .sort_values('customer_count', ascending=False)
)
res_5.display()

In [0]:
result_5.show(5)

In [0]:
# ── Tests for Problem 5 ─────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'country_code' in cols, "Missing column: country_code"
assert 'customer_count' in cols, "Missing column: customer_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 5 passed ✓  ({cnt} rows)")